In [75]:
from dotenv import load_dotenv
import os

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [76]:
%pip install -U langchain-groq
from langchain_groq import ChatGroq

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [77]:
#TOOL CREATION 

@tool 
def multiply(a:int, b:int) -> int:
    """Given 2 numbers a and b returns their product"""
    return a*b

In [78]:
print(multiply.invoke({'a' : 3, 'b' : 4}))

12


In [79]:
multiply.name
multiply.description
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [80]:
#TOOL BINDING

#creating llm 
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
)

#can bind multiple tools by adding comma
llm_with_tools = llm.bind_tools([multiply])
#whenever llm needs multiplication , it can call that function 

TOOL CALLING : 
is the process where llm decides during the convo that it needs to use a specific tool and generate a output with : name of the tool, and the argument to call with 

When we call llm it reads the input : 
if input = hi how are you -> then it will run normally 
but if input = multiply 8 by 7 -> then it will check all available tools and gets multiply tool 
                                prints name of the tool and generate the schema that what to pass in input 

In [81]:
llm_with_tools.invoke("hi how are you")
#in this we can see the content but no tool call

AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 230, 'total_tokens': 255, 'completion_time': 0.079969805, 'completion_tokens_details': None, 'prompt_time': 0.029383804, 'prompt_tokens_details': None, 'queue_time': 0.164758283, 'total_time': 0.109353609}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3bf7-93e4-7cc0-b690-ea69697e9a5e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 230, 'output_tokens': 25, 'total_tokens': 255})

In [82]:
#rather than sending query direct, converted first into human message and store it in the list of messages
#and later store the result into messages list

query = HumanMessage("Can you multiply 7 by 9 ")

messages = [query]

result = llm_with_tools.invoke(messages)
#in this we can see in output named tool call but no content 

messages.append(result)  

In [83]:
print(messages)

[HumanMessage(content='Can you multiply 7 by 9 ', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gxszzrmc4', 'function': {'arguments': '{"a":7,"b":9}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 235, 'total_tokens': 254, 'completion_time': 0.057263242, 'completion_tokens_details': None, 'prompt_time': 0.048547588, 'prompt_tokens_details': None, 'queue_time': 0.161463859, 'total_time': 0.10581083}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3bf7-953b-7590-bcd8-4af09f3ee941-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 9}, 'id': 'gxszzrmc4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 235, 'output_tokens': 19, 'total_tokens': 254})]


In [84]:
result.tool_calls[0]['args']
#can see name , arguments , unique id 

{'a': 7, 'b': 9}

The llm doesnt actually runs the tool , it only suggest the input and the tool (tool calling actual exectution  done by langchain)
Coz its dangerious if llm runs tool and give wrong answer by calling wrong tool 
LLM just advices which to use with these input parameters and give result to user 


TOOL EXECUTION : 
its the step where the actual python function(tool) is run using the input args that the llm suggested during tool calling 

ex : llm said call the multiply tool with a = 7 and b = 9
    TOOL EXECUTION is when you or LangChain actually run multiply(a=7,b=9) and get the result 

In [85]:
multiply.invoke(result.tool_calls[0]['args'])

63

In [86]:
tool_result = multiply.invoke(result.tool_calls[0])
#returns the tool message that can be send to llm 
#tool_result is also a type of message

In [87]:
messages.append(tool_result)

In [88]:
messages

[HumanMessage(content='Can you multiply 7 by 9 ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gxszzrmc4', 'function': {'arguments': '{"a":7,"b":9}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 235, 'total_tokens': 254, 'completion_time': 0.057263242, 'completion_tokens_details': None, 'prompt_time': 0.048547588, 'prompt_tokens_details': None, 'queue_time': 0.161463859, 'total_time': 0.10581083}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3bf7-953b-7590-bcd8-4af09f3ee941-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 9}, 'id': 'gxszzrmc4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 235, 'output_tokens': 19, 'total_tokens': 254}),
 ToolMessage(co

In [89]:
for m in messages:
    print(type(m))
    print(m)
    print("-" * 50)

<class 'langchain_core.messages.human.HumanMessage'>
content='Can you multiply 7 by 9 ' additional_kwargs={} response_metadata={}
--------------------------------------------------
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'tool_calls': [{'id': 'gxszzrmc4', 'function': {'arguments': '{"a":7,"b":9}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 235, 'total_tokens': 254, 'completion_time': 0.057263242, 'completion_tokens_details': None, 'prompt_time': 0.048547588, 'prompt_tokens_details': None, 'queue_time': 0.161463859, 'total_time': 0.10581083}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f3bf7-953b-7590-bcd8-4af09f3ee941-0' tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 9}, 'id': 'gxszzrmc4', 'type': 'tool_call'}] in

In [90]:
llm_with_tools.invoke(messages).content

#we will get in content this time -> the product of 7 by 9 is 63

'The result of multiplying 7 by 9 is 63.'